In [ ]:
%pip install kmodes gower hdbscan

EN ESTE NOTEBOOK:

- Categorizamos los scores.
- Creamos un nuevo dataset (df) en el que se han cambiado los errores y se han reagrupado las categóricas, e imputamos. (este dataset tiene que tener los valores brutos en los biomarcadores).
- Vemos cuál es la combinación de variables numéricas que maximiza el silhoutte score para k = 2, 3, 4, 5. (para esto hay que normalizar primero con StandardScaler).
- Para las tres mejores combinaciones, describimos estos grupos teniendo en cuenta vairables del embarazo, variables de eco y scores en formato categorías. (hay que incluir la media y la desviación para numéricas)
- Usamos las variables numéricas y las categóricas nuevas (las reagrupadas) para hacer clústering y ver la mejor combinación de variables que maximiza el silhouette score.
- Para las tres mejores combinaciones, describimos estos grupos teniendo en cuenta vairables del embarazo, variables de eco y scores en formato categorías. (hay que incluir la media y la desviación para numéricas y % para categóricas)

ADICIONALMENTE HACEMOS:
- Revisamos las variables musculo, grasa, etc que se habían tratado como categóricas
- Revisamos los missings en las variables que aparecen como nan.

Comparación de 2 grupos
- Prueba de normalidad. Hay normalidad si con shapiro da p < 0.05, |skewness| < 1 y |curtosis| <3.
- Para las variables numéricas con distribución normal, usamos prueba T para muestras independientes. Aquí habría que ver si las varianzas son homogéneas, con la prueba de Levene.
- Para las variables numéricas con distribución no normal usamos la prueba de U Mann-Whitney.
El resultado de lo anterior lo ponemos en formato: media +/- desviación estándar (para las que tengan distribución normal) y mediana +/- IQR para las variables con distribución no normal. 
- Para las variables categóricas ponemos el resultado en formato n (%) y usamos el Test de Fisher para variables de 2 categorías, y el chi square para variables con más categorías.

Comparación de > 2 grupos
- Para la normalidad. Hay normalidad si con shapiro da p < 0.05, |skewness| < 1 y |curtosis| <3.
- Para las vriables numéricas con distribución normal con varianzas homogéneas: ANOVA clásico y post-hoc con Tukey
- Para variables numéricas con distribución normal sin varianzas homogéneas: Welch ANOVA y post-hoc con Games-Howell
- Para variables numéricas con distribución no normal: Kruskal–Wallis y post-hoc con Dunn
- Para las variables categóricas ponemos el resultado en formato n (%) y usamos chi square.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Categorización de scores

In [ ]:
df_cat = pd.read_excel('variables_CARDIOMOM.xlsx')

scl_mapping = {
    'Nada': 0, 'Muy poco': 1, 'Poco': 2, 'Bastante': 3, 'Mucho': 4
}

# Funciones de categorización

def cat_regicor_dieta(score):
    if pd.isna(score): return np.nan
    if score < 23: return "Baja adherencia"
    elif 23 <= score <= 29: return "Adherencia moderada"
    else: return "Alta adherencia"

def cat_regicor_actividad(met):
    if pd.isna(met): return np.nan
    if met < 500: return "Actividad física baja"
    elif 500 <= met <= 1000: return "Actividad física moderada"
    else: return "Actividad física alta"

def cat_pss14(score):
    if pd.isna(score): return np.nan
    if score <= 13: return "Nivel de estrés bajo"
    elif 14 <= score <= 26: return "Nivel de estrés moderado"
    elif 27 <= score <= 40: return "Nivel de estrés alto"
    else: return "Nivel de estrés muy alto"

def cat_mfe30(score):
    if pd.isna(score): return np.nan
    if score < 8: return "Rendimiento de memoria óptimo"
    elif 9 <= score <= 35: return "Función normal (lapsus leves)"
    elif 36 <= score <= 50: return "Deterioro leve"
    else: return "Disfunción moderada-grave"

def cat_scl90r(t_score):
    if pd.isna(t_score): return np.nan
    if t_score >= 80: return "Patología severa"
    elif t_score >= 65: return "EN RIESGO"
    else: return "Normal / Sin riesgo"


df_cat['Cat_Dieta'] = df_cat['Score total'].apply(cat_regicor_dieta)
df_cat['Cat_Actividad'] = df_cat['Act física TOTAL (MET/semana)'].apply(cat_regicor_actividad)
df_cat['Cat_Estres'] = df_cat['Score'].apply(cat_pss14)
df_cat['Cat_Memoria'] = df_cat['Score.1'].apply(cat_mfe30)

# Procesamiento SCL-90-R
scl_items_cols = df_cat.loc[:, "Dolores de cabeza":"Pensar que en mi cabeza hay algo que no funciona bien"].columns
df_scl_numeric = df_cat[scl_items_cols].replace(scl_mapping).apply(pd.to_numeric, errors='coerce')
df_cat['SCL90_IGS_Raw'] = df_scl_numeric.mean(axis=1)

def calcular_t_score(row):
    raw = row['SCL90_IGS_Raw']
    if pd.isna(raw): return np.nan
    # Baremo Mujer (Media 0.17, DT 0.11) basado en el PDF
    return 50 + 10 * ((raw - 0.17) / 0.11)

df_cat['SCL90_T_Score'] = df_cat.apply(calcular_t_score, axis=1)
df_cat['Cat_SCL90R'] = df_cat['SCL90_T_Score'].apply(cat_scl90r)


cols_to_show = ['ID', 'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R']

print(df_cat[cols_to_show].head(20).to_string(index=False))

# Nuevo dataset + imputación
Con errores corregidos y categóricas agrupadas

In [ ]:
# Usamos 'datos_embarazo.csv' porque es el último csv que creamos antes de la primera imputación. Tiene todo el preprocesamiento al completo
df = pd.read_csv('datos_embarazo.csv')

In [ ]:
df.shape

In [ ]:
df.columns[1:50]

In [ ]:
df.columns[50:91]

Las columnas con las que trabajamos son: 

['id', 'fecha_firma_ci_cardiomom', 'fecha_firma_ci_muestbio',
       'fecha_nac', 'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest',
       'talla', 'imc_ini_gest', 'nivel_estudios', 'parto_previo_mayor37_pre',
       'parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg',
       'ant_obito', 'ant_pe', 'ant_hellp', 'ant_cesarea',
       'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido',
       'enf_autoinm', 'fuma', 'alcohol', 'drogas', 'fur_pre',
       'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'fecha_eco_1tri',
       'eg_eco_1tri', 'uterinas_p95_1tri', 'plgf_1tri', 'riesgo_pe_1tri',
       'uterinas_p95_eco_2tri', 'deter_sflt1_plgf_gest', 'fecha_ult_deter',
       'eg_deter_sflt1_plgf', 'valor_sflt1', 'valor_plgf', 'ratio_sflt1_plgf',
       'fecha_parto', 'eg_parto', 'ini_trabajo_parto_espontaneo', 'peso_rn',
       'apgar_1min', 'apgar_5min', 'apgar_10min',
       'hemorragia_pospart_transfusion', 'edema_agudo_pulmon', 'histerectomia',
       'otras', 'hipertension_gest', 'pe', 'sd_hellp',
       'desprendimiento_placenta', 'obito_fetal', 'hemocerebral_ictus',
       'embolia_tep', 'trombosis_venosa_prof', 'uci_materna_ucoi', 'covid',
       'cir', 'peg', 'diabetes_gest', 'colestasis_intrahepatica',
       'corioamnionitis', 'estudio_inicial_Angiocor', 'estudio_inicial_BiSC',
       'estudio_inicial_EUROPE', 'estudio_inicial_Ninguno',
       'etnia_Asia_Oriental', 'etnia_Blanca', 'etnia_Latina', 'etnia_Mixto',
       'etnia_Negra', 'etnia_Sureste_asiatico', 'concepcion_Espontanea',
       'concepcion_FIV', 'concepcion_FIV_ovodonacion',
       'concepcion_Inseminacion', 'tipo_parto_Cesarea', 'tipo_parto_Eutocico',
       'tipo_parto_Instrumentado', 'sexo_rn_Masculino', 'tomo_Aspirina',
       'tomo_Heparina', 'tomo_Antihipertensivo', 'tomo_Ninguna']

## Corrección de errores

In [ ]:
# Recuperamos las correcciones de notebooks pasados y añadimos nuevas

# Modificar el ID 15: fecha_eco_1tri y eg_eco_1tri
df.loc[df['id'] == 15, ['fecha_eco_1tri', 'eg_eco_1tri']] = ['2020-02-13', 12.5]

# Modificar el ID 392: fecha_eco_1tri y eg_eco_1tri
df.loc[df['id'] == 392, ['fecha_eco_1tri', 'eg_eco_1tri']] = ['2018-09-13', 13]

# Modificar el ID 568: fecha_eco_1tri y eg_eco_1tri
df.loc[df['id'] == 568, ['fecha_eco_1tri', 'eg_eco_1tri']] = ['2020-05-21', 12.9]

# Modificar el ID 647: fur_pre, fecha_eco_1tri, eg_eco_1tri y eg_parto
df.loc[df['id'] == 647, ['fur_pre','fecha_eco_1tri', 'eg_eco_1tri', 'eg_parto']] = ['2021-06-08','2021-08-31', 12, 41]

# Modificar el ID 55: fecha_parto y eg_parto
df.loc[df['id'] == 55, ['fecha_parto', 'eg_parto']] = ['2019-02-07', 37.6]

# Modificar el ID 383: fecha_parto y eg_parto
df.loc[df['id'] == 383, ['fecha_parto', 'eg_parto']] = ['2020-07-19', 38.9]

# Modificar el ID 585: fecha_parto y eg_parto
df.loc[df['id'] == 585, ['fecha_parto', 'eg_parto']] = ['2019-09-03', 40.5]

# Convertir a nan algunos valores de 'grasa_visceral'
ids_grasa_faltante = [98, 101, 182, 349, 359, 628]
df.loc[df['id'].isin(ids_grasa_faltante), 'grasa_visceral'] = np.nan

# Convertir a nan algunos valores de 'musculo'
ids_musculo_faltante = [98, 101, 103, 359]
df.loc[df['id'].isin(ids_musculo_faltante), 'musculo'] = np.nan

# Convertir a nan algunos valores de 'ratio_cintura_cadera'
df.loc[df['id'] == 311, 'ratio_cintura_cadera'] = np.nan

# Modificar el ID 11: 'diam_telediastolico'
df.loc[df['id'] == 11, 'diam_telediastolico'] = 37

# Modificar el ID 189: 'diam_telesistolico'
df.loc[df['id'] == 189, 'diam_telesistolico'] = 35

# Modificar el ID 189: 'dtsvi_indexado'
df.loc[df['id'] == 189, 'dtsvi_indexado'] = 22.73

# Modificar algunos valores de 'septo_iv_diastole'
df.loc[df['id'] == 308, 'septo_iv_diastole'] = 7.7
df.loc[df['id'] == 325, 'septo_iv_diastole'] = 7.0
df.loc[df['id'] == 466, 'septo_iv_diastole'] = 8.7
df.loc[df['id'] == 467, 'septo_iv_diastole'] = 8.7
df.loc[df['id'] == 469, 'septo_iv_diastole'] = 9.5
df.loc[df['id'] == 615, 'septo_iv_diastole'] = 8.8

# Modificar algunos valores de 'pared_posterior_vi_diastole'
df.loc[df['id'] == 308, 'pared_posterior_vi_diastole'] = 6.9
df.loc[df['id'] == 325, 'pared_posterior_vi_diastole'] = 6.9
df.loc[df['id'] == 466, 'pared_posterior_vi_diastole'] = 7.2
df.loc[df['id'] == 467, 'pared_posterior_vi_diastole'] = 6.0
df.loc[df['id'] == 469, 'pared_posterior_vi_diastole'] = 9.5
df.loc[df['id'] == 615, 'pared_posterior_vi_diastole'] = 8.3

# Modificar el ID 149: 'diam_ai'
df.loc[df['id'] == 149, 'diam_ai'] = 34

# Modificar algunos valores de 'diam_raiz_aortica'
df.loc[df['id'] == 186, 'diam_raiz_aortica'] = 31
df.loc[df['id'] == 189, 'diam_raiz_aortica'] = 28
df.loc[df['id'] == 278, 'diam_raiz_aortica'] = np.nan
df.loc[df['id'] == 444, 'diam_raiz_aortica'] = 28
df.loc[df['id'] == 452, 'diam_raiz_aortica'] = 26
df.loc[df['id'] == 511, 'diam_raiz_aortica'] = 27
df.loc[df['id'] == 524, 'diam_raiz_aortica'] = 23.7

# Modificar algunos valores de 'diam_raiz_aortica_indexada'
df.loc[df['id'] == 186, 'diam_raiz_aortica_indexada'] = 15.27
df.loc[df['id'] == 189, 'diam_raiz_aortica_indexada'] = 18.18
df.loc[df['id'] == 278, 'diam_raiz_aortica_indexada'] = np.nan
df.loc[df['id'] == 440, 'diam_raiz_aortica_indexada'] = 13.41
df.loc[df['id'] == 452, 'diam_raiz_aortica_indexada'] = 12.83
df.loc[df['id'] == 511, 'diam_raiz_aortica_indexada'] = 14.85
df.loc[df['id'] == 524, 'diam_raiz_aortica_indexada'] = 14.22

# Modificar algunos valores de 'tsvi'
df.loc[df['id'] == 127, 'tsvi'] = 17.7
df.loc[df['id'] == 136, 'tsvi'] = 19.3
df.loc[df['id'] == 304, 'tsvi'] = 19.6
df.loc[df['id'] == 408, 'tsvi'] = 22.0
df.loc[df['id'] == 445, 'tsvi'] = 18.2
df.loc[df['id'] == 505, 'tsvi'] = 18.5
df.loc[df['id'] == 621, 'tsvi'] = 18.6

# Modificar el ID 554: 'ai_volumen'
df.loc[df['id'] == 554, 'ai_volumen'] = 84.7

# Modificar el ID 349: 'ad_volumen'
df.loc[df['id'] == 349, 'ad_volumen'] = 20.9

# Modificar algunos valores de 'tapse' 
df.loc[df['id'] == 36, 'tapse'] = 21.1
df.loc[df['id'] == 191, 'tapse'] = 25
df.loc[df['id'] == 291, 'tapse'] = 35
df.loc[df['id'] == 412, 'tapse'] = 29.0

# Modificar algunos valores de 'onda_s_anillo_tricuspideo'
df.loc[df['id'] == 238, 'onda_s_anillo_tricuspideo'] = np.nan
df.loc[df['id'] == 543, 'onda_s_anillo_tricuspideo'] = 13.3

# Modificar algunos valores de 'diam_basal_vd'
df.loc[df['id'] == 95, 'diam_basal_vd'] = 38.0
df.loc[df['id'] == 186, 'diam_basal_vd'] = 41.0

# Modificar algunos valores de 'e_mitral'
df.loc[df['id'] == 337, 'e_mitral'] = 58.7
df.loc[df['id'] == 546, 'e_mitral'] = 109.6

# Modificar el ID 381: 'tiempo_deceleracion_vm'
df.loc[df['id'] == 381, 'tiempo_deceleracion_vm'] = 90

# Modificar algunos valores de 'e_mitral_lateral'
df.loc[df['id'] == 343, 'e_mitral_lateral'] = 6.3
df.loc[df['id'] == 488, 'e_mitral_lateral'] = 8.5

# Modificar algunos valores de 'e_mitral_medial'
df.loc[df['id'] == 343, 'e_mitral_medial'] = 8.4
df.loc[df['id'] == 488, 'e_mitral_medial'] = 6.4
df.loc[df['id'] == 522, 'e_mitral_medial'] = 8.8

# Modificar algunos valores de 'e_e'
df.loc[df['id'] == 337, 'e_e'] = 3.645
df.loc[df['id'] == 546, 'e_e'] = 5.5776

# Modificar algunos valores de 'gc_tsvi'
df.loc[df['id'] == 100, 'gc_tsvi'] = 5.2
df.loc[df['id'] == 169, 'gc_tsvi'] = 5.7
df.loc[df['id'] == 366, 'gc_tsvi'] = 4.5
df.loc[df['id'] == 445, 'gc_tsvi'] = 5.1
df.loc[df['id'] == 502, 'gc_tsvi'] = 4.9

# Modificar el ID 300: 'left_pulsability_index'
df.loc[df['id'] == 300, 'left_pulsability_index'] = 1.55

# Modificar el ID 392: 'urato_acidourico'
df.loc[df['id'] == 392, 'urato_acidourico'] = 4.4

# Modificar algunos valores de 'prote_totales_orina'
df.loc[df['id'] == 2, 'prote_totales_orina'] = np.nan
df.loc[df['id'] == 4, 'prote_totales_orina'] = np.nan
df.loc[df['id'] == 36, 'prote_totales_orina'] = np.nan
df.loc[df['id'] == 101, 'prote_totales_orina'] = 0.07
df.loc[df['id'] == 191, 'prote_totales_orina'] = 0.07
df.loc[df['id'] == 192, 'prote_totales_orina'] = 0.07
df.loc[df['id'] == 193, 'prote_totales_orina'] = 0.08
df.loc[df['id'] == 194, 'prote_totales_orina'] = 0.14
df.loc[df['id'] == 197, 'prote_totales_orina'] = 0.07
df.loc[df['id'] == 199, 'prote_totales_orina'] = 0.10
df.loc[df['id'] == 202, 'prote_totales_orina'] = 0.13
df.loc[df['id'] == 203, 'prote_totales_orina'] = 0.10

In [ ]:
# Eliminar las variables de fechas y determinaciones
columnas_a_eliminar = [
    'fecha_firma_ci_cardiomom', 
    'fecha_firma_ci_muestbio', 
    'fecha_nac', 
    'fur_pre', 
    'fecha_eco_1tri', 
    'fecha_ult_deter', 
    'fecha_parto',
    'uterinas_p95_1tri', 
    'plgf_1tri', 
    'uterinas_p95_eco_2tri', 
    'deter_sflt1_plgf_gest',
    'Unnamed: 0'
]

# Eliminar las columnas del DataFrame
df = df.drop(columns=columnas_a_eliminar, errors='ignore')

# Opcional: Verificar las columnas restantes
print("Columnas actuales en el DataFrame:")
print(df.columns.tolist())

## Reagrupación de categóricas

In [ ]:
# Reagrupamos categóricas

# CONTADORES DE ANTECENDENTES obstétricos vs médicos
# Lo que hacemos es contar el número de antecedentes que tiene una paciente
# ant_obstetrico ('parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp')
cols_ant_obstetrico = [
    'parto_previo_menor37_pre', 'aborto_menor20', 
    'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp'
]
df['ant_obstetrico'] = df[cols_ant_obstetrico].sum(axis=1)

# ant_medico ('ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas')
cols_ant_medico = [
    'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 
    'enf_autoinm', 'fuma', 'alcohol', 'drogas'
]
df['ant_medico'] = df[cols_ant_medico].sum(axis=1)

# Aplicamos StandardScaler porque son categóricas ordinales (más es peor) 
scaler = StandardScaler()
df[['ant_obstetrico', 'ant_medico']] = scaler.fit_transform(df[['ant_obstetrico', 'ant_medico']])


# COMPLICACIONES DURANTE LA GESTACIÓN (variables nominales 0/1)

# Trastorno hipertensivo: 1 si pe, sd_hellp o 'hipertension_gest' vs 0 si no tiene ninguna. 
cols_hipertension = ['pe', 'sd_hellp', 'hipertension_gest']
df['trastorno_hipertensivo'] = df[cols_hipertension].max(axis=1)

# Complicación placentaria: 'cir', 'peg', 'desprendimiento_placenta'
cols_placentaria = ['cir', 'peg', 'desprendimiento_placenta']
df['comp_placentaria'] = df[cols_placentaria].max(axis=1)

# Complicación materna grave: 'uci_materna_ucoi', 'hemocerebral_ictus', 'trombosis_venosa_prof'
cols_materna_grave = ['uci_materna_ucoi', 'hemocerebral_ictus', 'trombosis_venosa_prof']
df['comp_maternaGrave'] = df[cols_materna_grave].max(axis=1)

# Complicación metabólica: 'diabetes_gest', 'colestasis_intrahepatica'
cols_metabolica = ['diabetes_gest', 'colestasis_intrahepatica']
df['comp_metabolica'] = df[cols_metabolica].max(axis=1)


# LIMPIEZA
# Eliminamoa variables que tengan 0 casos positivos
n_positivos = df.sum(numeric_only=True)
cols_a_eliminar = n_positivos[n_positivos == 0].index
df.drop(columns=cols_a_eliminar, inplace=True)

print(f"Variables agrupadas. Se han eliminado {len(cols_a_eliminar)} columnas con cero positivos.")

In [ ]:
print(cols_a_eliminar)

En df no aparecen las variables 'grasa_visceral', 'musculo' ni 'ratio_cintura_cadera', por lo que no entendemos por qué se eliminan

In [ ]:
# Eliminamos las variables que fueron agrupadas
cols_originales_agrupadas = list(set(
    cols_ant_obstetrico + 
    cols_ant_medico + 
    cols_hipertension + 
    cols_placentaria + 
    cols_materna_grave + 
    cols_metabolica
))

# Usamos errors='ignore' por si alguna columna ya fue eliminada en el paso de '0 positivos'
df.drop(columns=cols_originales_agrupadas, inplace=True, errors='ignore')

print(f"Se han eliminado las {len(cols_originales_agrupadas)} variables originales para simplificar el modelo.")

## Imputamos

In [ ]:
# COMPROBACIÓN DE QUE TODAS LAS VARIABLES SON NUMÉRICAS
vars_numericas = df.select_dtypes(include=['number']).columns.tolist()

tabla_numericas = pd.DataFrame(vars_numericas, columns=['Variables Numéricas'])

print("TABLA DE NUMÉRICAS")
print(tabla_numericas)

In [ ]:
df.shape

In [ ]:
numericas = ['id', 'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
            'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 'valor_sflt1', 
            'eg_deter_sflt1_plgf', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'eg_eco_1tri', 'valor_plgf', 
            'ratio_sflt1_plgf']

In [ ]:
ordinales = ['nivel_estudios', 'riesgo_pe_1tri', 'ant_medico', 'ant_obstetrico']

In [ ]:
# Import basics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [ ]:
# Para asegurar que el imputador funcione bien convertimos los valores faltantes, representados en blanco, a NaN
df.replace("", np.nan, inplace=True)

In [ ]:
min_values_dict = df.min().to_dict()
max_values_dict = df.max().to_dict()

In [ ]:
min_values_dict

In [ ]:
# Definimos cada variable según su tipo
df_num = df[numericas]
df_cat_ord = df[ordinales]

In [ ]:
# Probar esto para ver que los min/max son correctos
min_values_num = [min_values_dict[col] for col in df_num.columns]
max_values_num = [max_values_dict[col] for col in df_num.columns]
min_values_num

In [ ]:
min_values_all = [min_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all = [max_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all

In [ ]:
# Utilizamos el código para imputar sin hacer división train/test
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(
        n_estimators=num_estimators,
        random_state=random_state,
        max_depth=m_depth,
        min_samples_leaf=m_samples_leaf
    )
    imputer = IterativeImputer(
        estimator=tree,
        random_state=random_state,
        max_iter=max_iter,
        min_value=min_value,
        max_value=max_value,
        imputation_order='ascending',
        initial_strategy='mean',
        verbose=2
    )
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline_all(data_num, data_cat,
                            num_estimators, m_depth, m_samples_leaf,
                            max_iter, random_state=1234,
                            guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in data_num.columns]
    max_values_num = [max_values_dict[col] for col in data_num.columns]

    min_values_all = [min_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]
    max_values_all = [max_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]

    # Imputación de numéricas
    imputer_num = _missForest_fit(
        data_num, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_num,
        max_value=max_values_num,
        random_state=random_state
    )

    data_num_imp = pd.DataFrame(
        _missForest_transform(imputer_num, data_num),
        columns=data_num.columns,
        index=data_num.index
    )

    # Unir con categóricas (ordinales)
    data_all = pd.concat([data_num_imp, data_cat], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(
        data_all, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_all,
        max_value=max_values_all,
        random_state=random_state
    )

    data_all_imp = _missForest_transform(imputer_all, data_all)

    # Control de variables categóricas (ordinales)
    for i in range(data_num.shape[1], data_all.shape[1]):
        data_all_imp[:, i] = np.clip(data_all_imp[:, i], min_values_all[i], max_values_all[i])

    data_final = pd.DataFrame(data_all_imp, columns=data_all.columns, index=data_all.index)

    for col in data_cat.columns:
        data_final[col] = data_final[col].round().astype(int)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return data_final

In [ ]:
data_final = imputacion_pipeline_all(
    data_num=df_num,
    data_cat=df_cat_ord,   # aquí SOLO ordinales (codificadas como enteros)
    num_estimators=50,
    m_depth=4,
    m_samples_leaf=3,
    max_iter=100,
    random_state=1234,
    guardar_imputador=True)

In [ ]:
df.shape

In [ ]:
df.columns[1:45]

In [ ]:
df.columns[46:91]

In [ ]:
nominales_post_onehot = ['covid',
                'edema_agudo_pulmon', 'hemorragia_pospart_transfusion', 
                'ini_trabajo_parto_espontaneo', 
                'parto_previo_mayor37_pre',
                'estudio_inicial_Angiocor', 'estudio_inicial_BiSC', 'estudio_inicial_EUROPE', 'estudio_inicial_Ninguno', 
                'etnia_Asia_Oriental', 'etnia_Blanca', 'etnia_Latina', 'etnia_Mixto', 'etnia_Negra', 'etnia_Sureste_asiatico', 
                'concepcion_Espontanea', 'concepcion_FIV', 'concepcion_FIV_ovodonacion', 'concepcion_Inseminacion', 
                'tipo_parto_Cesarea', 'tipo_parto_Eutocico', 'tipo_parto_Instrumentado', 
                'sexo_rn_Masculino',
                'tomo_Aspirina', 'tomo_Heparina', 'tomo_Antihipertensivo', 'tomo_Ninguna',
                'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica']

                
df_cat_nom = df[nominales_post_onehot]

In [ ]:
def impute_nominal_simple(X_nom):
    imp = SimpleImputer(strategy='most_frequent')
    X_nom_imp = pd.DataFrame(imp.fit_transform(X_nom), columns=X_nom.columns, index=X_nom.index)
    return X_nom_imp, imp

In [ ]:
X_nom_imp, imp = impute_nominal_simple(df_cat_nom)

In [ ]:
df_imp = pd.concat([data_final, X_nom_imp], axis=1)

In [ ]:
# Comprobar que no hay valores faltantes
df_imp.isna().sum().sum()

In [ ]:
df_imp

In [ ]:
df_imp.to_csv('imputado.csv')

# Mejor combinación de variables para maximizar silhouette score en clústering (de reunion7)

Las variables consideradas aquí son:
['peso_ini_gest', 'peso_fin_gest’, 'aumento_peso_gest', 'talla', 'imc_ini_gest',  'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 'sflt1_MoM’, 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM']

In [ ]:
# Como se consideran las variables 'plgf_MoM', 'ratio_MoM' y 'sflt1_MoM' y hasta ahora estábamos trabajando con los biomarcadores,
# tenemos que aplicar la transformación de MoM a estas variables.

def calcular_mom_variables(df):
    df = df.copy()
    # Definimos las condiciones basadas en 'eg_deter_sflt1_plgf'
    condiciones = [
        (df['eg_deter_sflt1_plgf'] >= 10) & (df['eg_deter_sflt1_plgf'] < 15), # 10 a 14.99
        (df['eg_deter_sflt1_plgf'] >= 15) & (df['eg_deter_sflt1_plgf'] < 20), # 15 a 19.99
        (df['eg_deter_sflt1_plgf'] >= 20) & (df['eg_deter_sflt1_plgf'] < 24),
        (df['eg_deter_sflt1_plgf'] >= 24) & (df['eg_deter_sflt1_plgf'] < 29),
        (df['eg_deter_sflt1_plgf'] >= 29) & (df['eg_deter_sflt1_plgf'] < 34),
        (df['eg_deter_sflt1_plgf'] >= 34) & (df['eg_deter_sflt1_plgf'] < 37),
        (df['eg_deter_sflt1_plgf'] >= 37)
]

    # Divisores para cada columna
    div_ratio = [24.8, 10.5, 4.92, 3.06, 3.75, 9.03, 19.6]
    div_sflt1 = [1328, 1355, 1299, 1355, 1742, 2552, 3485]
    div_plgf  = [52.6, 135, 264, 465, 471, 284, 191]

    # Aplicamos np.select para obtener el divisor correspondiente a cada fila
    # Si no cumple ninguna condición, devuelve NaN (np.nan)
    df['ratio_MoM'] = df['ratio_sflt1_plgf'] / np.select(condiciones, div_ratio, default=np.nan)
    df['sflt1_MoM'] = df['valor_sflt1'] / np.select(condiciones, div_sflt1, default=np.nan)
    df['plgf_MoM']  = df['valor_plgf'] / np.select(condiciones, div_plgf, default=np.nan)

    return df

df_imp_clust = calcular_mom_variables(df_imp)

variables_a_borrar = ['ratio_sflt1_plgf', 'valor_sflt1', 'valor_plgf']
df_imp_clust = df_imp_clust.drop(columns=variables_a_borrar, errors='ignore')

In [ ]:
# Estaba saliendo un error porque las variables MoM del id 448 salían vacías. Las calculamos a mano

# Aseguramos que la columna sea numérica
df_imp['eg_deter_sflt1_plgf'] = pd.to_numeric(df_imp['eg_deter_sflt1_plgf'], errors='coerce')

mask_448 = df_imp_clust['id'] == 448
if df_imp_clust.loc[mask_448, 'sflt1_MoM'].isnull().any():
    print("Corrigiendo ID 448 manualmente...")
    df_imp_clust.loc[mask_448, 'ratio_MoM'] = df_imp_clust.loc[mask_448, 'ratio_sflt1_plgf'] / 9.03
    df_imp_clust.loc[mask_448, 'sflt1_MoM'] = df_imp_clust.loc[mask_448, 'valor_sflt1'] / 2552
    df_imp_clust.loc[mask_448, 'plgf_MoM']  = df_imp_clust.loc[mask_448, 'valor_plgf'] / 284

# Verificamos
print(df_imp_clust.loc[df_imp_clust['id'] == 448, ['sflt1_MoM', 'plgf_MoM', 'ratio_MoM']])

## Clústering

In [ ]:
# Seleccionar variables numéricas para el clustering
numericas_cols = ['peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
                  'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 'sflt1_MoM', 
                  'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM']

# Dejamos la columna 'id' fuera del escalado
features_clustering = [col for col in numericas_cols if col != 'id']

df_cluster = df_imp_clust[numericas_cols + ['id']].copy()
df_cluster = df_cluster.dropna(subset=numericas_cols)  # dropna solo en features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[features_clustering])

# Convertir el resultado a dataframe
df_scaled = pd.DataFrame(X_scaled, columns=features_clustering, index=df_cluster.index)

df_scaled['id'] = df_cluster['id'].values              # id viaja con df_scaled y df_combi
df_scaled.head()

In [ ]:
# Aplicamos K-Means para distintos valores de k y calculamos métricas
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

k_values = range(2, 11)
silhouette_scores = []
calinski_scores = []
davies_scores = []

# Fijamos random_state por reproducibilidad
random_state = 1234

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(df_scaled)
    
    silhouette_scores.append(silhouette_score(df_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(df_scaled, labels))
    davies_scores.append(davies_bouldin_score(df_scaled, labels))

# Mostramos los resultados en un DataFrame
metrics_df = pd.DataFrame({
    'k': k_values,
    'Silhouette': silhouette_scores,
    'Calinski-Harabasz': calinski_scores,
    'Davies-Bouldin': davies_scores
})

metrics_df

## Buscamos la mejor combinación

Explicación del código (porque no se prueba con todas las variables porque es un número inmenso de combinaciones):

Se empieza con un conjunto vacío y añade la variable que, combinada con las anteriores, saca el mayor Silhouette Score. Se deja de añadir variables cuando el score deja de subir. Esto evita incluir variables "ruido" que diluyen la separación de los clústeres.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df_combi = df_scaled.copy()
# Excluimos 'id' y variables no numéricas de la lista de candidatas
candidatas = [v for v in numericas_cols if v.lower() != 'id']

def find_top_3_subsets(data, k_range):
    all_k_results = []
    
    for k in k_range:
        print(f"Buscando las 3 mejores combinaciones para k={k}...")
        
        # Lista para guardar todas las combinaciones probadas para este k
        combinations_tested = []
        current_subset = []
        remaining_vars = list(candidatas)
        
        # Iniciamos la selección hacia adelante (Forward Selection)
        while remaining_vars:
            step_scores = []
            
            for var in remaining_vars:
                test_subset = current_subset + [var]
                
                # Evaluamos solo si hay 2 o más variables
                if len(test_subset) >= 2:
                    X = StandardScaler().fit_transform(data[test_subset])
                    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
                    labels = kmeans.fit_predict(X)
                    
                    score = silhouette_score(X, labels)
                    
                    # Guardamos el resultado de esta combinación específica
                    res = {
                        'k': k,
                        'silhouette': score,
                        'num_variables': len(test_subset),
                        'variables': list(test_subset)
                    }
                    combinations_tested.append(res)
                    step_scores.append((score, var))
            
            # Caso especial: primera variable (no hay score con n=1)
            if not step_scores:
                current_subset.append(remaining_vars.pop(0))
                continue
            
            # Ordenamos para encontrar la mejor variable de este paso y seguir creciendo
            step_scores.sort(key=lambda x: x[0], reverse=True)
            best_var = step_scores[0][1]
            current_subset.append(best_var)
            remaining_vars.remove(best_var)

        # Para cada k, ordenamos todas las combinaciones probadas y extraemos el Top 3
        df_k = pd.DataFrame(combinations_tested)
        top_3 = df_k.sort_values(by='silhouette', ascending=False).head(3)
        all_k_results.append(top_3)
    
    # Consolidamos resultados en un único DataFrame
    return pd.concat(all_k_results, ignore_index=True)

# Ejecutamos la búsqueda para k = 2, 3, 4, 5
top_3_results = find_top_3_subsets(df_combi, [2, 3, 4, 5])

# Mostrar resultados
print("\nTOP 3 COMBINACIONES QUE MAXIMIZAN EL SILHOUETTE SCORE")
pd.set_option('display.max_colwidth', None) # Para ver todas las variables
display(top_3_results)

# Descripción grupos (variables numéricas)

In [ ]:
df_eco = pd.read_csv('datos.csv')

In [ ]:
# DESCRIPCIÓN DE GRUPOS — MEJORES COMBINACIONES (CLUSTERING MIXTO)
# Variables numéricas + categóricas agrupadas
# Descripción en función de: variables de gestación, ecocardiografía y scores

# VARIABLES DE DESCRIPCIÓN
#    Gestación y eco: las presentes en df_eco
#    MoM: presentes en df_imp_clust
#    Scores: calculados al principio del notebook (df_cat)

# Recuperamos antecedentes en escala original (sin StandardScaler) 
df_tmp_raw = pd.read_csv('datos_embarazo.csv')
cols_ant_obstetrico = ['parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp']
cols_ant_medico     = ['ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas']
df['ant_obstetrico_raw'] = df_tmp_raw[cols_ant_obstetrico].sum(axis=1)
df['ant_medico_raw']     = df_tmp_raw[cols_ant_medico].sum(axis=1)


vars_comp_y_ant = ['trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave',
                   'comp_metabolica', 'ant_obstetrico_raw', 'ant_medico_raw']

vars_embarazo = [c for c in [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'eg_eco_1tri', 'eg_parto',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min'
] if c in df_imp.columns]

vars_eco = [c for c in [
    'diam_telediastolico', 'diam_telesistolico', 'dtsvi_indexado', 'septo_iv_diastole',
    'pared_posterior_vi_diastole', 'diam_ai', 'ai_volumen', 'ad_volumen', 'tapse',
    'e_mitral', 'e_e', 'gc_tsvi', 'urato_acidourico'
] if c in df_eco.columns]

vars_biomarcadores = [c for c in ['ratio_MoM', 'sflt1_MoM', 'plgf_MoM'] if c in df_imp_clust.columns]

# CONSTRUCCIÓN DEL DATAFRAME
# Unimos df_imp_clust (MoMs) con df_eco (eco) y con df_cat (scores)

df['id'] = df['id'].astype(float)
df_imp_clust['id'] = df_imp_clust['id'].astype(float)

# Eco: unir usando 'id'
df_maestro = pd.merge(
    df_imp_clust,
    df_eco[vars_eco + ['id']],
    on='id',
    how='left'
)

# Scores categóricos
cols_cat_scores = ['ID', 'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R']
df_maestro = pd.merge(
    df_maestro,
    df_cat[cols_cat_scores],
    left_on='id',
    right_on='ID',
    how='left'
)

# Antecedentes
df_maestro = pd.merge(
    df_maestro,
    df[['id'] + vars_comp_y_ant],
    on='id',
    how='left'
)

# FUNCIÓN DE DESCRIPCIÓN
#    Numéricas: media ± desviación típica
#    Categóricas: frecuencia relativa (%)

def describir_clusters(df, variables_num, variables_cat):
    partes = []
    if variables_num:
        res_num = df.groupby('cluster')[variables_num].agg(
            lambda x: f"{x.mean():.2f} ± {x.std():.2f}"
        ).T
        res_num.columns = [f"Grupo {i}" for i in res_num.columns]
        partes.append(res_num)

    if variables_cat:
        cat_list = []
        for col in variables_cat:
            valid = df[[col, 'cluster']].dropna(subset=[col])
            if valid.empty:
                continue
            ct_n = pd.crosstab(valid[col], valid['cluster'])
            ct_pct = pd.crosstab(valid[col], valid['cluster'], normalize='columns') * 100
            
            ct_formateada = ct_n.copy().astype(str)
            for col_cl in ct_n.columns:
                ct_formateada[col_cl] = [f"{n} ({p:.1f}%)" for n, p in zip(ct_n[col_cl], ct_pct[col_cl])]
                
            ct_formateada.index = [f"{col}: {idx}" for idx in ct_formateada.index]
            cat_list.append(ct_formateada)
            
        if cat_list:
            res_cat = pd.concat(cat_list)
            res_cat.columns = [f"Grupo {i}" for i in res_cat.columns]
            partes.append(res_cat)

    return pd.concat(partes) if partes else pd.DataFrame()


# GRUPOS DE VARIABLES

grupos_informe = {
    "GESTACIÓN Y BIOMARCADORES": vars_embarazo + vars_biomarcadores,
    "ECOCARDIOGRAFÍA": vars_eco,
    "SCORES CATEGÓRICOS": [c for c in cols_cat_scores if c != 'ID' and c in df_maestro.columns]
}

# VARIABLES QUE NECESITAN ESCALADO EN EL CLUSTERING MIXTO

variables_a_escalar_mixto = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto',
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM',
    'ratio_MoM', 'ant_obstetrico', 'ant_medico'
]


# BUCLE — UNA TABLA POR COMBINACIÓN DEL TOP-3 MIXTO

pd.set_option('display.max_colwidth', None)

for idx, row in top_3_results.iterrows():
    k_actual        = row['k']
    vars_clustering = row['variables']
    sil_score       = row['silhouette']

    print(f"COMBINACIÓN {idx + 1}  |  K = {k_actual}  |  SILHOUETTE = {sil_score:.4f}")
    print(f"Variables de clustering: {vars_clustering}\n")

    # Clustering sobre df_imp_clust  
    ids_limpios = df_cluster['id'].values   # ids que ya pasaron por dropna en la búsqueda
    X_orig = df_imp_clust[df_imp_clust['id'].isin(ids_limpios)][vars_clustering].copy()

    escalar_ahora = [v for v in vars_clustering if v in variables_a_escalar_mixto]
    if escalar_ahora:
        X_orig[escalar_ahora] = StandardScaler().fit_transform(X_orig[escalar_ahora])

    kmeans = KMeans(n_clusters=k_actual, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_orig.values)

    # Unir clusters a df_maestro mediante 'id' 
    mapa    = pd.DataFrame({'id': ids_limpios, 'cluster': labels})
    df_desc = df_maestro.drop(columns=['cluster'], errors='ignore')
    df_desc = pd.merge(df_desc, mapa, on='id', how='inner')

    # Tamaño de cada grupo
    tamanos = df_desc['cluster'].value_counts().sort_index()
    tamanos.index = [f"Grupo {i}" for i in tamanos.index]
    print("\nTAMAÑO DE LOS GRUPOS")
    print(tamanos.to_frame(name='N').T.to_string())

    # Tablas descriptivas por bloque
    for nombre_bloque, vars_bloque in grupos_informe.items():
        v_num = [
            v for v in vars_bloque
            if v in df_desc.columns and pd.api.types.is_numeric_dtype(df_desc[v])
        ]
        v_cat = [
            v for v in vars_bloque
            if v in df_desc.columns and not pd.api.types.is_numeric_dtype(df_desc[v])
        ]

        if v_num or v_cat:
            print(f"\n{nombre_bloque}")
            tabla = describir_clusters(df_desc, v_num, v_cat)
            display(tabla)

    print()


# Clústering con variables numéricas + categóricas agrupadas

Las variables consideradas aquí son:
['peso_ini_gest', 'peso_fin_gest’, 'aumento_peso_gest', 'talla', 'imc_ini_gest',  'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 'sflt1_MoM’, 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 'ratio_MoM', 'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica']

In [ ]:
# A las numéricas se les aplica StandardScaler
# A las categóricas ordinales (ant_medico, ant_obstetrico) se les aplica StandardScaler
# A las nominales no hay que aplicarles One Hot porque solo hay dos posibles valores

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.impute import SimpleImputer

todas_variables = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 
    'ratio_MoM', 'ant_obstetrico', 'ant_medico', 'trastorno_hipertensivo', 
    'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica'
]

variables_a_escalar = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 
    'ratio_MoM', 'ant_obstetrico', 'ant_medico'
]


# Incluimos 'id' y eliminamos filas con NaN solo en las variables de clustering
X = df_imp_clust[todas_variables + ['id']].copy()
X = X.dropna(subset=todas_variables)

scaler = StandardScaler()
X[variables_a_escalar] = scaler.fit_transform(X[variables_a_escalar])

In [ ]:
# Análisis de K-Means y métricas para k de 2 a 10
resultados = []

print(f"{'k':<5} | {'Silhouette':<12} | {'Davies-Bouldin':<15} | {'Calinski-Harabasz':<18}")
print("-" * 60)

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    
    # Cálculo de las tres métricas principales
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    resultados.append({
        'k': k, 
        'Silhouette': sil, 
        'Davies-Bouldin': db, 
        'Calinski-Harabasz': ch
    })
    
    print(f"{k:<5} | {sil:<12.4f} | {db:<15.4f} | {ch:<18.4f}")

# Convertir a DataFrame para facilitar la visualización o exportación
df_resultados_clustering = pd.DataFrame(resultados)

# Mejor combinación de variables para maximizar silhouette score en clústering (de reunion8)

In [ ]:
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from gower import gower_matrix
import pandas as pd
import numpy as np

# Definimos las candidatas excluyendo explícitamente el 'id'
candidatas = [v for v in X.columns if v.lower() not in ['id', 'unnamed: 0']]

variables_a_escalar = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto', 
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM', 
    'ratio_MoM', 'ant_obstetrico', 'ant_medico'
]

cols_cat = ['ant_obstetrico_bin', 'ant_medico_bin', 'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 'comp_metabolica', 'ant_obstetrico', 'ant_medico']

def find_top_3_subsets_mixed(data, k_range):
    all_k_results = []
    
    for k in k_range:
        print(f"Buscando las mejores combinaciones para k={k}...")
        combinations_tested = []
        current_subset = []
        remaining_vars = list(candidatas)
        
        while remaining_vars:
            step_scores = []
            for var in remaining_vars:
                test_subset = current_subset + [var]
                
                if len(test_subset) >= 2:
                    X_test = data[test_subset].copy()
                    
                    escalar_ahora = [v for v in test_subset if v in variables_a_escalar]
                    if escalar_ahora:
                        X_test[escalar_ahora] = StandardScaler().fit_transform(X_test[escalar_ahora])
                    
                    # Categóricas
                    cat_indices = []
                    cat_bools = []
                    for i, c in enumerate(X_test.columns):
                        if c in cols_cat:
                            cat_indices.append(i)
                            cat_bools.append(True)
                            X_test[c] = X_test[c].astype(str)
                        else:
                            cat_bools.append(False)
                    
                    # Gower Matrix
                    try:
                        X_gower = gower_matrix(X_test, cat_features=cat_bools).astype(np.float64)
                    except Exception:
                        X_gower = gower_matrix(X_test).astype(np.float64)
                        
                    if len(cat_indices) == 0:
                        from sklearn.cluster import KMeans
                        kmeans = KMeans(n_clusters=k, n_init=1, max_iter=5, random_state=42)
                        labels = kmeans.fit_predict(X_test.values)
                    else:
                        kp = KPrototypes(n_clusters=k, init='Cao', n_init=1, max_iter=5, verbose=0, random_state=42)
                        labels = kp.fit_predict(X_test.values, categorical=cat_indices)
                    
                    if len(set(labels)) > 1:
                        score = silhouette_score(X_gower, labels, metric='precomputed')
                    else:
                        score = -1
                    
                    res = {
                        'k': k,
                        'silhouette': score,
                        'num_variables': len(test_subset),
                        'variables': list(test_subset)
                    }
                    combinations_tested.append(res)
                    step_scores.append((score, var))
            
            if not step_scores:
                current_subset.append(remaining_vars.pop(0))
                continue
                
            step_scores.sort(key=lambda x: x[0], reverse=True)
            best_var = step_scores[0][1]
            current_subset.append(best_var)
            remaining_vars.remove(best_var)

        df_k = pd.DataFrame(combinations_tested)
        if not df_k.empty:
            top_3 = df_k.sort_values(by='silhouette', ascending=False).head(3)
            all_k_results.append(top_3)
    
    return pd.concat(all_k_results, ignore_index=True)

top_3_results_mixto = find_top_3_subsets_mixed(X, range(2, 6))

print("\nTOP 3 COMBINACIONES POR K")
pd.set_option('display.max_colwidth', None)
display(top_3_results_mixto)


# Descripción grupos (categóricas)

In [ ]:
# DESCRIPCIÓN DE GRUPOS — MEJORES COMBINACIONES (CLUSTERING MIXTO)
# Variables numéricas + categóricas agrupadas
# Descripción en función de: variables de gestación, ecocardiografía y scores

# VARIABLES DE DESCRIPCIÓN
#    Gestación y eco: las presentes en df_eco
#    MoM: presentes en df_imp_clust
#    Scores: calculados al principio del notebook (df_cat)

vars_embarazo = [c for c in [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'eg_eco_1tri', 'eg_parto',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min'
] if c in df_imp.columns]

vars_eco = [c for c in [
    'diam_telediastolico', 'diam_telesistolico', 'dtsvi_indexado', 'septo_iv_diastole',
    'pared_posterior_vi_diastole', 'diam_ai', 'ai_volumen', 'ad_volumen', 'tapse',
    'e_mitral', 'e_e', 'gc_tsvi', 'urato_acidourico'
] if c in df_eco.columns]

vars_biomarcadores = [c for c in ['ratio_MoM', 'sflt1_MoM', 'plgf_MoM'] if c in df_imp_clust.columns]

# CONSTRUCCIÓN DEL DATAFRAME
# Unimos df_imp_clust (MoMs) con df_eco (eco) y con df_cat (scores)

df['id'] = df['id'].astype(float)
df_imp_clust['id'] = df_imp_clust['id'].astype(float)

# Eco: unir usando 'id'
df_maestro = pd.merge(
    df_imp_clust,
    df_eco[vars_eco + ['id']],
    on='id',
    how='left'
)

# Scores categóricos
cols_cat_scores = ['ID', 'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R']
df_maestro = pd.merge(
    df_maestro,
    df_cat[cols_cat_scores],
    left_on='id',
    right_on='ID',
    how='left'
)

# FUNCIÓN DE DESCRIPCIÓN
#    Numéricas: media ± desviación típica
#    Categóricas: frecuencia relativa (%)

def describir_clusters(df, variables_num, variables_cat):
    partes = []
    if variables_num:
        res_num = df.groupby('cluster')[variables_num].agg(
            lambda x: f"{x.mean():.2f} ± {x.std():.2f}"
        ).T
        res_num.columns = [f"Grupo {i}" for i in res_num.columns]
        partes.append(res_num)

    if variables_cat:
        cat_list = []
        for col in variables_cat:
            valid = df[[col, 'cluster']].dropna(subset=[col])
            if valid.empty:
                continue
            ct_n = pd.crosstab(valid[col], valid['cluster'])
            ct_pct = pd.crosstab(valid[col], valid['cluster'], normalize='columns') * 100
            
            ct_formateada = ct_n.copy().astype(str)
            for col_cl in ct_n.columns:
                ct_formateada[col_cl] = [f"{n} ({p:.1f}%)" for n, p in zip(ct_n[col_cl], ct_pct[col_cl])]
                
            ct_formateada.index = [f"{col}: {idx}" for idx in ct_formateada.index]
            cat_list.append(ct_formateada)
            
        if cat_list:
            res_cat = pd.concat(cat_list)
            res_cat.columns = [f"Grupo {i}" for i in res_cat.columns]
            partes.append(res_cat)

    return pd.concat(partes) if partes else pd.DataFrame()


# GRUPOS DE VARIABLES

grupos_informe = {
    "GESTACIÓN Y BIOMARCADORES": vars_embarazo + vars_biomarcadores,
    "ECOCARDIOGRAFÍA": vars_eco,
    "SCORES CATEGÓRICOS": [c for c in cols_cat_scores if c != 'ID' and c in df_maestro.columns]
}

# VARIABLES QUE NECESITAN ESCALADO EN EL CLUSTERING MIXTO

variables_a_escalar_mixto = [
    'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest',
    'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'eg_parto',
    'sflt1_MoM', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'plgf_MoM',
    'ratio_MoM', 'ant_obstetrico', 'ant_medico'
]


# BUCLE — UNA TABLA POR COMBINACIÓN DEL TOP-3 MIXTO

pd.set_option('display.max_colwidth', None)

for idx, row in top_3_results_mixto.iterrows():
    k_actual        = row['k']
    vars_clustering = row['variables']
    sil_score       = row['silhouette']

    print(f"COMBINACIÓN {idx + 1}  |  K = {k_actual}  |  SILHOUETTE = {sil_score:.4f}")
    print(f"Variables de clustering: {vars_clustering}\n")

    # Clustering sobre df_imp_clust  
    ids_limpios = X['id'].values   # ids que ya pasaron por dropna en la búsqueda
    X_orig = df_imp_clust[df_imp_clust['id'].isin(ids_limpios)][vars_clustering].copy()

    escalar_ahora = [v for v in vars_clustering if v in variables_a_escalar_mixto]
    if escalar_ahora:
        X_orig[escalar_ahora] = StandardScaler().fit_transform(X_orig[escalar_ahora])

    kmeans = KMeans(n_clusters=k_actual, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_orig.values)

    # Unir clusters a df_maestro mediante 'id' 
    mapa    = pd.DataFrame({'id': ids_limpios, 'cluster': labels})
    df_desc = df_maestro.drop(columns=['cluster'], errors='ignore')
    df_desc = pd.merge(df_desc, mapa, on='id', how='inner')

    # Tamaño de cada grupo
    tamanos = df_desc['cluster'].value_counts().sort_index()
    tamanos.index = [f"Grupo {i}" for i in tamanos.index]
    print("\nTAMAÑO DE LOS GRUPOS")
    print(tamanos.to_frame(name='N').T.to_string())

    # Tablas descriptivas por bloque
    for nombre_bloque, vars_bloque in grupos_informe.items():
        v_num = [
            v for v in vars_bloque
            if v in df_desc.columns and pd.api.types.is_numeric_dtype(df_desc[v])
        ]
        v_cat = [
            v for v in vars_bloque
            if v in df_desc.columns and not pd.api.types.is_numeric_dtype(df_desc[v])
        ]

        if v_num or v_cat:
            print(f"\n{nombre_bloque}")
            tabla = describir_clusters(df_desc, v_num, v_cat)
            display(tabla)

    print()


In [ ]:
# ANÁLISIS DE MISSINGS EN df_maestro

todas_las_vars = (
    vars_embarazo + vars_biomarcadores +
    vars_comp_y_ant +
    vars_eco +
    [c for c in cols_cat_scores if c != 'id' and c in df_maestro.columns]
)

# Tabla global de missings 
resumen_missing = []
for col in todas_las_vars:
    if col not in df_maestro.columns:
        resumen_missing.append({'Variable': col, 'N total': '-', 'N missing': '-',
                                '% missing': '-', 'Presente en df_maestro': False})
        continue
    n_total   = len(df_maestro)
    n_missing = df_maestro[col].isna().sum()
    pct       = n_missing / n_total * 100
    resumen_missing.append({
        'Variable': col,
        'N total': n_total,
        'N missing': n_missing,
        '% missing': f"{pct:.1f}%",
        'Presente en df_maestro': True
    })

df_missing = pd.DataFrame(resumen_missing)
df_missing = df_missing.set_index('Variable')

print("MISSINGS GLOBALES EN df_maestro")
display(df_missing)

# Variables con algún missing 
problematicas = df_missing[
    (df_missing['Presente en df_maestro'] == True) &
    (df_missing['% missing'] != '0.0%')
].copy()

print(f"\n  Variables con algún missing: {len(problematicas)} de {len(df_missing)}")
display(problematicas[['N missing', '% missing']])

Las variables de eco salen con NaN porque hemos tomado estas variables de 'datos.csv', que no está imputado. Son los valores Nan que venían desde el principio.

# Análisis estadístico

In [ ]:
# Definición ampliada de grupos para el informe
grupos_informe_ampliado = {
    "GESTACIÓN Y BIOMARCADORES": grupos_informe["GESTACIÓN Y BIOMARCADORES"], 
    "ECOCARDIOGRAFÍA": grupos_informe["ECOCARDIOGRAFÍA"],
    "SCORES CATEGÓRICOS": [
        'Cat_Dieta', 'Cat_Actividad', 'Cat_Estres', 'Cat_Memoria', 'Cat_SCL90R'
    ],
    
    "TA, CARÓTIDA Y OFTÁLMICA": [
        'ta_sistolica', 'ta_diastolica', 'frec_cardiaca', 
        'right_1peak_systolic_velocity', 'right_2peak_systolic_velocity', 
        'right_pulsatility_index', 'right_psv_ratio', 
        'left_1peak_systolic_velocity', 'left_2peak_systolic_velocity', 
        'left_pulsatility_index', 'left_psv_ratio', 'right_mean', 'left_mean'
    ],
    
    "ANALÍTICA": [
        'plgf_pg', 'troponina_t', 'nt_probnp', 'hemoglobina', 'hematocrito', 
        'leucocitos', 'plaquetas', 'glucosa', 'sodio', 'potasio', 
        'urato_acidourico', 'creatinina', 'hemoglobina_glicada', 'ast', 'alt', 
        'bilirubina_total', 'ldh', 'vldl', 'ldl', 'hdl', 'colesterol_total', 
        'trigliceridos', 'prote_totales_orina', 'albumina_orina', 
        'ratio_albumina_creatinina', 'tirotropina', 'prolactina'
    ],
    
    "ANTECEDENTES": [
        'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 
        'comp_metabolica', 'covid', 'COVID', 'ant_obstetrico_raw', 'ant_medico_raw',
        'ant_obstetrico_bin', 'ant_medico_bin', 'ant_obstetrico', 'ant_medico',
        'parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp',
        'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas'
    ]
}

In [ ]:
from scipy import stats
from itertools import combinations
import pandas as pd

def realizar_analisis_avanzado(df, var, grupo_col):
    """
    Versión final: 
    - Test global según normalidad/varianza.
    - Test post-hoc (dos a dos) con corrección de Bonferroni.
    - Manejo robusto de variables categóricas y numéricas.
    """
    grupos_ids = sorted(df[grupo_col].unique())
    dict_datos = {g: df[df[grupo_col] == g][var].dropna() for g in grupos_ids}
    n_grupos = len(grupos_ids)
    
    datos_validos = [d for d in dict_datos.values() if len(d) > 0]
    if len(datos_validos) < 2:
        return [var, "N/A", "N/A", "N/A"] + ["-"] * n_grupos

    es_numerica = pd.api.types.is_numeric_dtype(df[var]) and not set(df[var].dropna().unique()).issubset({0, 1})

    # A. TEST TOTAL
    if es_numerica:
        p_shapiro = stats.shapiro(df[var].dropna())[1] if len(df[var].dropna()) > 3 else 0
        skew = df[var].skew()
        kurt = df[var].kurtosis()
        es_normal = (p_shapiro > 0.05) and (abs(skew) < 1) and (abs(kurt) < 3)
        p_levene = stats.levene(*datos_validos)[1] if len(datos_validos) > 1 else 1
        homogeneas = p_levene > 0.05

        if n_grupos == 2:
            test_name = "Prueba T" if es_normal else "U Mann-Whitney"
            g1, g2 = list(dict_datos.values())[0], list(dict_datos.values())[1]
            if es_normal:
                _, p_val_total = stats.ttest_ind(g1, g2, equal_var=homogeneas)
            else:
                _, p_val_total = stats.mannwhitneyu(g1, g2)
        else:
            if es_normal:
                test_name = "ANOVA" if homogeneas else "Welch ANOVA"
                p_val_total = stats.f_oneway(*datos_validos)[1] if homogeneas else stats.alexandergovern(*datos_validos).pvalue
            else:
                test_name = "Kruskal-Wallis"
                _, p_val_total = stats.kruskal(*datos_validos)
        metrica = "media ± DE" if es_normal else "mediana ± IQR"
    else:
        contingencia = pd.crosstab(df[var], df[grupo_col])
        if n_grupos == 2 and len(df[var].dropna().unique()) == 2:
            test_name = "Fisher"
            _, p_val_total = stats.fisher_exact(contingencia)
        else:
            test_name = "Chi-Square"
            _, p_val_total, _, _ = stats.chi2_contingency(contingencia)

    # B. P-VALORES DOS A DOS
    pairs = list(combinations(grupos_ids, 2))
    p_pairs = []
    for g1, g2 in pairs:
        d1, d2 = dict_datos[g1], dict_datos[g2]
        if len(d1) > 0 and len(d2) > 0:
            if es_numerica:
                _, p_p = stats.mannwhitneyu(d1, d2)
            else:
                sub_cont = pd.crosstab(df[df[grupo_col].isin([g1, g2])][var], df[df[grupo_col].isin([g1, g2])][grupo_col])
                _, p_p, _, _ = stats.chi2_contingency(sub_cont)
            p_adj = min(p_p * len(pairs), 1.0)
            p_pairs.append(f"G{g1}vG{g2}: {p_adj:.3f}")
        else:
            p_pairs.append(f"G{g1}vG{g2}: N/A")
    pairwise_str = " | ".join(p_pairs)

    # C. RESULTADOS POR GRUPO
    res_grupos = []
    for g in grupos_ids:
        g_data = dict_datos[g]
        if len(g_data) == 0:
            res_grupos.append("N/A")
        elif es_numerica:
            if metrica == "media ± DE":
                res_grupos.append(f"{g_data.mean():.2f} ± {g_data.std():.2f}")
            else:
                res_grupos.append(f"{g_data.median():.2f} ± {g_data.quantile(0.75)-g_data.quantile(0.25):.2f}")
        else:
            counts = g_data.value_counts().sort_index()
            total_n = len(g_data)
            res_grupos.append(" | ".join([f"{c}: {int(n)} ({n/total_n*100:.1f}%)" for c, n in counts.items()]))
            
    return [var, test_name, f"{p_val_total:.4f}", pairwise_str] + res_grupos

## Análisis estadístico de los grupos del clústering de numéricas

In [ ]:
# Usamos los resultados correspondientes (puedes cambiarlo a top_3_results_mixto en la celda 68)
df_analisis = top_3_results 

for idx, row in df_analisis.iterrows():
    if True:
        k_actual = int(row['k'])
        vars_actual = row['variables']
        
        print(f"ANÁLISIS PARA COMBINACIÓN {idx + 1} (SILHOUETTE: {row['silhouette']:.4f})")
        print(f"Variables clustering: {vars_actual}")

        # Ejecutar el clustering para asignar labels
        X_cl = df_combi[vars_actual].values
        kmeans = KMeans(n_clusters=k_actual, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X_cl)

        # Merge de datos: Unimos labels con maestro y con df_eco
        mapa = pd.DataFrame({'id': df_combi['id'].values, 'cluster_temp': labels})
        df_desc = pd.merge(df_maestro, mapa, on='id', how='inner')
        # Cruce con df_eco para obtener analíticas y TA
        df_desc = pd.merge(df_desc, df_eco, on='id', how='left', suffixes=('', '_eco'))

        # Aseguramos que las variables existan en df_desc antes de pasar a la estadística
        df_tmp_raw = pd.read_csv('datos_embarazo.csv') # Cargamos el original
        cols_obst = ['parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp']
        cols_med = ['ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas']
        
        # Calculamos los raw directamente en el dataframe de descripción
        df_tmp_raw['ant_obstetrico_raw'] = df_tmp_raw[cols_obst].sum(axis=1)
        df_tmp_raw['ant_medico_raw'] = df_tmp_raw[cols_med].sum(axis=1)
        
        # Añadimos los raw en el dataframe de descripción
        df_desc = pd.merge(df_desc, df_tmp_raw[['id', 'ant_obstetrico_raw', 'ant_medico_raw']], on='id', how='left')
        
        # Generar tablas por cada bloque de variables definido arriba
        for nombre, variables in grupos_informe_ampliado.items():
            print(f"\n>>> {nombre}")
            data_rows = []
            
            for v in variables:
                if v in df_desc.columns and v not in ['id', 'ID']:
                    res = realizar_analisis_avanzado(df_desc, v, 'cluster_temp')
                    if res:
                        data_rows.append(res)
            
            # Definir encabezados dinámicos
            headers = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"] + [f"G{i}" for i in range(k_actual)]
            
            if data_rows:
                df_final = pd.DataFrame(data_rows, columns=headers)
                display(df_final)
    else:
        print(f"Combinación {idx + 1} saltada (Silhouette {row['silhouette']:.4f} <= 0.55)")

## Análisis estadístico de los grupos del clústering de numéricas + categóricas

In [ ]:
df_resultados_mixto = top_3_results_mixto 

for idx, row in df_resultados_mixto.iterrows():
    # FILTRO: Solo silhouette > 0.55
    if True:
        k_actual = int(row['k'])
        vars_actual = row['variables']
        
        print(f"ANÁLISIS PARA COMBINACIÓN {idx + 1} (SILHOUETTE: {row['silhouette']:.4f})")
        print(f"Variables clustering: {vars_actual}")

        # Preparar datos para el clustering
        # El clustering debe hacerse sobre los mismos datos que generaron el silhouette
        X_cl = X[vars_actual].values
        kmeans = KMeans(n_clusters=k_actual, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X_cl)

        # Merge de datos: Labels + Maestro + Eco (para analíticas/TA)
        mapa = pd.DataFrame({'id': X['id'].values, 'cluster_temp': labels})
        df_desc = pd.merge(df_maestro, mapa, on='id', how='inner')
        # Añadimos df_eco para tener las variables de TA y Analítica
        df_desc = pd.merge(df_desc, df_eco, on='id', how='left', suffixes=('', '_eco'))

        # Aseguramos que las variables existan en df_desc antes de pasar a la estadística
        df_tmp_raw = pd.read_csv('datos_embarazo.csv') # Cargamos el original
        cols_obst = ['parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp']
        cols_med = ['ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas']
        
        # Calculamos los raw directamente en el dataframe de descripción
        df_tmp_raw['ant_obstetrico_raw'] = df_tmp_raw[cols_obst].sum(axis=1)
        df_tmp_raw['ant_medico_raw'] = df_tmp_raw[cols_med].sum(axis=1)
        
        # Unimos esos cálculos a nuestro dataframe de análisis por ID
        df_desc = pd.merge(df_desc, df_tmp_raw[['id', 'ant_obstetrico_raw', 'ant_medico_raw']], on='id', how='left')

        # Procesar por bloques de variables (usando el diccionario ampliado)
        for nombre, variables in grupos_informe_ampliado.items():
            print(f"\n>>> {nombre}")
            data_rows = []
            for v in variables:
                if v in df_desc.columns and v not in ['id', 'ID']:
                    res = realizar_analisis_avanzado(df_desc, v, 'cluster_temp')
                    if res:
                        data_rows.append(res)
            
            # Definimos los headers dinámicamente según k_actual
            headers = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"] + [f"G{i}" for i in range(k_actual)]
            
            if data_rows:
                df_final = pd.DataFrame(data_rows, columns=headers)
                display(df_final)
    else:
        print(f"Combinación {idx + 1} saltada (Silhouette {row['silhouette']:.4f} <= 0.55)")

El p-valor responde siempre a la pregunta de si son los grupos del clustering estadísticamente diferentes entre sí en una variable determinada.

La hipótesis nula en todos los casos es que no hay diferencia entre los grupos. Un p-valor bajo (< 0.05) indica que los grupos del clústring son significativamente distintos.

# Análisis de missings en variables usadas en la descripción de los grupos

In [ ]:
# Definición de las listas de variables
vars_ta_car_oft = [
    'ta_sistolica', 'ta_diastolica', 'frec_cardiaca', 
    'right_1peak_systolic_velocity', 'right_2peak_systolic_velocity', 
    'right_pulsatility_index', 'right_psv_ratio', 
    'left_1peak_systolic_velocity', 'left_2peak_systolic_velocity', 
    'left_pulsatility_index', 'left_psv_ratio', 'right_mean', 'left_mean'
]

vars_analitica = [
    'plgf_pg', 'troponina_t', 'nt_probnp', 'hemoglobina', 'hematocrito', 
    'leucocitos', 'plaquetas', 'glucosa', 'sodio', 'potasio', 'urato_acidourico', 
    'creatinina', 'hemoglobina_glicada', 'ast', 'alt', 'bilirubina_total', 
    'ldh', 'vldl', 'ldl', 'hdl', 'colesterol_total', 'trigliceridos', 
    'prote_totales_orina', 'albumina_orina', 'ratio_albumina_creatinina', 
    'tirotropina', 'prolactina'
]

vars_antecedentes = [
    'trastorno_hipertensivo', 'comp_placentaria', 'comp_maternaGrave', 
    'comp_metabolica', 'COVID', 'ant_obstetrico_raw', 'ant_medico_raw'
]

# Unión de datos para el análisis
df_missing_check = pd.merge(df_maestro, df_eco, on='id', how='left', suffixes=('', '_eco'))

def analizar_missings(df, lista_vars, nombre_grupo):
    # Filtrar solo las que existen en el dataframe
    existentes = [v for v in lista_vars if v in df.columns]
    
    missing_data = df[existentes].isnull().sum().to_frame(name='N_Missings')
    missing_data['Porcentaje (%)'] = (missing_data['N_Missings'] / len(df)) * 100
    missing_data['Grupo'] = nombre_grupo
    return missing_data

# Calcular missings y combinar
stats_ta = analizar_missings(df_missing_check, vars_ta_car_oft, 'TA/Oftálmica')
stats_ana = analizar_missings(df_missing_check, vars_analitica, 'Analítica')
stats_ant = analizar_missings(df_missing_check, vars_antecedentes, 'Antecedentes')

all_missing_stats = pd.concat([stats_ta, stats_ana, stats_ant])

# Preparar la tabla para mostrar (ordenar por mayor porcentaje de missings)
all_missing_stats = all_missing_stats.reset_index().rename(columns={'index': 'Variable'})
all_missing_stats = all_missing_stats.sort_values('Porcentaje (%)', ascending=False)

print(f"RESUMEN DE VALORES PERDIDOS POR GRUPO (Total pacientes: {len(df_missing_check)})")
display(all_missing_stats.style.format({
    'Porcentaje (%)': '{:.2f}%'
}).hide(axis='index')) # Ocultamos el índice numérico para que se vea más limpio

# NUEVO BLOQUE DE CLUSTERING AVANZADO (DATOS MIXTOS)

In [ ]:
import sys
import types
import numpy as np
import pandas as pd
import ast
from gower import gower_matrix
from hdbscan import HDBSCAN
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("----- INICIO BLOQUE AVANZADO -----")

# Definir KMedoids_Polyfill si no está disponible sklearn_extra
class KMedoids_Polyfill:
    def __init__(self, n_clusters=8, metric='euclidean', method='pam', init='heuristic', max_iter=300, random_state=None):
        self.n_clusters = n_clusters
        self.metric = metric
        self.method = method
        self.max_iter = max_iter
        self.random_state = random_state

    def fit_predict(self, X, y=None):
        rs = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        medoids = rs.choice(n_samples, self.n_clusters, replace=False)
        labels = np.zeros(n_samples, dtype=int)
        
        for _ in range(self.max_iter):
            distances = X[:, medoids]
            new_labels = np.argmin(distances, axis=1)
            new_medoids = np.zeros(self.n_clusters, dtype=int)
            for k in range(self.n_clusters):
                cluster_points = np.where(new_labels == k)[0]
                if len(cluster_points) == 0:
                    new_medoids[k] = medoids[k]
                    continue
                intra_distances = X[np.ix_(cluster_points, cluster_points)]
                costs = np.sum(intra_distances, axis=1)
                best_idx = np.argmin(costs)
                new_medoids[k] = cluster_points[best_idx]
                
            if np.array_equal(labels, new_labels) and np.array_equal(medoids, new_medoids):
                break
            labels = new_labels
            medoids = new_medoids
            
        self.labels_ = labels
        return labels

try:
    from sklearn_extra.cluster import KMedoids
except ImportError:
    print("Inyectando KMedoids desde polyfill debido a error de compilación...")
    if 'sklearn_extra' not in sys.modules:
        sk_ex = types.ModuleType('sklearn_extra')
        sk_ex_cl = types.ModuleType('sklearn_extra.cluster')
        sk_ex_cl.KMedoids = KMedoids_Polyfill
        sk_ex.cluster = sk_ex_cl
        sys.modules['sklearn_extra'] = sk_ex
        sys.modules['sklearn_extra.cluster'] = sk_ex_cl
    from sklearn_extra.cluster import KMedoids

# Transformación de antecedentes
df_tmp_raw = pd.read_csv('datos_embarazo.csv')
cols_obst = ['parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp']
cols_med = ['ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas']

df_tmp_raw['ant_obstetrico_raw'] = df_tmp_raw[cols_obst].sum(axis=1)
df_tmp_raw['ant_medico_raw'] = df_tmp_raw[cols_med].sum(axis=1)

df_tmp_raw['ant_obstetrico_bin'] = (df_tmp_raw['ant_obstetrico_raw'] > 0).astype(int)
df_tmp_raw['ant_medico_bin'] = (df_tmp_raw['ant_medico_raw'] > 0).astype(int)

if 'ant_obstetrico_bin' in df_maestro.columns:
    df_maestro = df_maestro.drop(columns=['ant_obstetrico_bin', 'ant_medico_bin'])
df_maestro = pd.merge(df_maestro, df_tmp_raw[['id', 'ant_obstetrico_bin', 'ant_medico_bin']], on='id', how='left')


# --- DESGLOSE POR K PARA PAM Y K-PROTOTYPES ---
k_range = [2, 3, 4, 5]

for algo in ['K-Medoids (PAM)', 'K-Prototypes']:
    for k_val in k_range:
        # Extraer las combinaciones correspondientes a este k de top_3_results_mixto
        df_k_combs = top_3_results_mixto[top_3_results_mixto['k'] == k_val]
        
        resultados_k = []
        
        for comb_idx, row_comb in df_k_combs.iterrows():
            mejor_vars_str = row_comb['variables']
            if isinstance(mejor_vars_str, str):
                mejor_vars = ast.literal_eval(mejor_vars_str)
            else:
                mejor_vars = list(mejor_vars_str)
                
            vars_clustering_adv = []
            for v in mejor_vars:
                if v == 'ant_obstetrico':
                    vars_clustering_adv.append('ant_obstetrico_bin')
                elif v == 'ant_medico':
                    vars_clustering_adv.append('ant_medico_bin')
                else:
                    vars_clustering_adv.append(v)

            if 'ant_obstetrico_bin' not in vars_clustering_adv: vars_clustering_adv.append('ant_obstetrico_bin')
            if 'ant_medico_bin' not in vars_clustering_adv: vars_clustering_adv.append('ant_medico_bin')
            
            df_clust = df_maestro[['id'] + vars_clustering_adv].dropna().copy()
            if len(df_clust) < 10:
                continue

            ids_clust = df_clust['id'].values
            X_features = df_clust[vars_clustering_adv].copy()

            cols_cat = ['ant_obstetrico_bin', 'ant_medico_bin']
            cat_bools = [c in cols_cat for c in X_features.columns]
            cat_indices = [i for i, c in enumerate(X_features.columns) if c in cols_cat]

            for c in cols_cat:
                if c in X_features.columns:
                    X_features[c] = X_features[c].astype('category')
                    
            try:
                X_gower = gower_matrix(X_features, cat_features=cat_bools).astype(np.float64)
            except Exception:
                X_gower = gower_matrix(X_features).astype(np.float64)
            
            if algo == 'K-Medoids (PAM)':
                kmed = KMedoids(n_clusters=k_val, metric='precomputed', method='pam', random_state=42)
                labels = kmed.fit_predict(X_gower)
                
                if len(set(labels)) > 1:
                    sil = silhouette_score(X_gower, labels, metric='precomputed')
                    resultados_k.append({'Combinacion': vars_clustering_adv, 'silhouette': sil, 'labels': labels, 'ids': ids_clust})
            
            else: # K-Prototypes
                X_kp = X_features.copy()
                for c in cols_cat:
                    if c in X_kp.columns:
                        X_kp[c] = X_kp[c].astype(str)
                
                kp = KPrototypes(n_clusters=k_val, init='Cao', n_init=2, verbose=0, random_state=42)
                labels = kp.fit_predict(X_kp.values, categorical=cat_indices)
                if len(set(labels)) > 1:
                    sil = silhouette_score(X_gower, labels, metric='precomputed')
                    resultados_k.append({'Combinacion': vars_clustering_adv, 'silhouette': sil, 'labels': labels, 'ids': ids_clust})
        
        if not resultados_k:
            continue
            
        df_res_k = pd.DataFrame(resultados_k).sort_values('silhouette', ascending=False)
        # Asegurar combinaciones unicas en caso de empates raros
        df_res_k['comb_str'] = df_res_k['Combinacion'].apply(str)
        df_res_k = df_res_k.drop_duplicates('comb_str').head(3)
        
        print(f"\n============================================================")
        print(f"RESULTADOS {algo} | k={k_val}")
        print(f"============================================================")
        
        # Mostrar tabla resumen de top 5
        resumen_top5 = df_res_k[['Combinacion', 'silhouette']].copy()
        resumen_top5['Rank'] = range(1, len(resumen_top5) + 1)
        display(resumen_top5[['Rank', 'Combinacion', 'silhouette']])
        
        # Describir Clusters para cada combinacion Top
        for rank_idx, (_, row_top) in enumerate(df_res_k.iterrows(), 1):
            print(f"\n---> Algoritmo: {algo} | k={k_val} | Combinación Top {rank_idx}")
            print(f"Silhouette = {row_top['silhouette']:.4f}")
            print(f"Variables: {row_top['Combinacion']}")
            
            mapa = pd.DataFrame({'id': row_top['ids'], 'cluster': row_top['labels']})
            mapa = mapa[mapa['cluster'] != -1]
            
            df_desc = df_maestro.drop(columns=['cluster'], errors='ignore')
            df_desc = pd.merge(df_desc, mapa, on='id', how='inner')
            
            print("TAMAÑO DE LOS GRUPOS")
            tam = df_desc['cluster'].value_counts().sort_index()
            tam.index = [f"Grupo {i}" for i in tam.index]
            print(tam.to_frame(name='N').T.to_string())

    for nombre_bloque, vars_bloque in grupos_informe_ampliado.items():
            v_num = [v for v in vars_bloque if v in df_desc.columns and pd.api.types.is_numeric_dtype(df_desc[v])]
            v_cat = [v for v in vars_bloque if v in df_desc.columns and not pd.api.types.is_numeric_dtype(df_desc[v])]
            
            # Forzar CUALQUIER variable relacionada con antecedentes a ser categórica (v_cat)
            to_remove = []
            for v in v_num:
                if 'ant_' in v.lower() or 'antecedente' in v.lower() or v == 'fuma' or v == 'alcohol' or v == 'drogas':
                    to_remove.append(v)
            for v in to_remove:
                v_num.remove(v)
                if v not in v_cat:
                    v_cat.append(v)
                    
            if v_num or v_cat:
                print(f"
{'='*40}")
                print(f">>> {nombre_bloque}")
                print(f"{'='*40}")
                
                # Tabla Descriptiva
                print("
[1] Tabla Descriptiva:")
                tabla_desc = describir_clusters(df_desc, v_num, v_cat)
                display(tabla_desc)
                
                # Tabla Estadística
                print("
[2] Análisis Estadístico Completo (P-valores dos a dos):")
                data_rows = []
                for v in (v_num + v_cat):
                    res = realizar_analisis_avanzado(df_desc, v, 'cluster')
                    if res:
                        data_rows.append(res)
                
                if data_rows:
                    k_actual_table = len(df_desc['cluster'].unique())
                    headers = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"] + [f"G{i}" for i in sorted(df_desc['cluster'].unique())]
                    df_estad = pd.DataFrame(data_rows, columns=headers)
                    display(df_estad)


# --- HDBSCAN ---
min_cluster_sizes = [5, 10, 15]
hdbscan_resultados = []

for _, row_comb in top_3_results_mixto.iterrows():
    mejor_vars_str = row_comb['variables']
    if isinstance(mejor_vars_str, str):
        mejor_vars = ast.literal_eval(mejor_vars_str)
    else:
        mejor_vars = list(mejor_vars_str)

    vars_clustering_adv = []
    for v in mejor_vars:
        if v == 'ant_obstetrico':
            vars_clustering_adv.append('ant_obstetrico_bin')
        elif v == 'ant_medico':
            vars_clustering_adv.append('ant_medico_bin')
        else:
            vars_clustering_adv.append(v)

    if 'ant_obstetrico_bin' not in vars_clustering_adv: vars_clustering_adv.append('ant_obstetrico_bin')
    if 'ant_medico_bin' not in vars_clustering_adv: vars_clustering_adv.append('ant_medico_bin')

    df_clust = df_maestro[['id'] + vars_clustering_adv].dropna().copy()
    if len(df_clust) < 10:
        continue
    ids_clust = df_clust['id'].values
    X_features = df_clust[vars_clustering_adv].copy()

    cols_cat = ['ant_obstetrico_bin', 'ant_medico_bin']
    cat_bools = [c in cols_cat for c in X_features.columns]

    for c in cols_cat:
        if c in X_features.columns:
            X_features[c] = X_features[c].astype('category')

    try:
        X_gower = gower_matrix(X_features, cat_features=cat_bools).astype(np.float64)
    except Exception:
        X_gower = gower_matrix(X_features).astype(np.float64)

    for mcs in min_cluster_sizes:
        try:
            hdb = HDBSCAN(metric='precomputed', min_cluster_size=mcs)
            labels = hdb.fit_predict(X_gower)
            mask = labels != -1
            if len(set(labels[mask])) > 1:
                X_clean = X_gower[mask][:, mask]
                sil = silhouette_score(X_clean, labels[mask], metric='precomputed')
                res_k = len(set(labels[mask]))
                hdbscan_resultados.append({
                    'min_size': mcs, 
                    'k_generado': res_k,
                    'silhouette': sil, 
                    'labels': labels, 
                    'ids': ids_clust,
                    'Combinacion': vars_clustering_adv
                })
        except Exception:
            pass

if hdbscan_resultados:
    df_hdbscan = pd.DataFrame(hdbscan_resultados).sort_values('silhouette', ascending=False)
    df_hdbscan['comb_str'] = df_hdbscan['Combinacion'].apply(str)
    # Seleccionamos las 5 mejores combinaciones y evitamos duplicadas
    df_hdbscan_top5 = df_hdbscan.drop_duplicates(subset=['silhouette', 'k_generado', 'comb_str']).head(3)

    print(f"\n============================================================")
    print(f"RESULTADOS HDBSCAN")
    print(f"============================================================")

    resumen_top5_hdb = df_hdbscan_top5[['min_size', 'k_generado', 'Combinacion', 'silhouette']].copy()
    resumen_top5_hdb['Rank'] = range(1, len(resumen_top5_hdb) + 1)
    display(resumen_top5_hdb[['Rank', 'min_size', 'k_generado', 'Combinacion', 'silhouette']])

    for rank_idx, (_, row_top) in enumerate(df_hdbscan_top5.iterrows(), 1):
        print(f"\n---> Algoritmo: HDBSCAN | min_size={row_top['min_size']} | k={row_top['k_generado']} | Combinación Top {rank_idx}")
        print(f"Silhouette = {row_top['silhouette']:.4f}")
        print(f"Variables: {row_top['Combinacion']}")

        mapa = pd.DataFrame({'id': row_top['ids'], 'cluster': row_top['labels']})
        mapa = mapa[mapa['cluster'] != -1] # filtrar ruido

        df_desc = df_maestro.drop(columns=['cluster'], errors='ignore')
        df_desc = pd.merge(df_desc, mapa, on='id', how='inner')

        print("TAMAÑO DE LOS GRUPOS (sin contar ruido '-1')")
        tam = df_desc['cluster'].value_counts().sort_index()
        tam.index = [f"Grupo {i}" for i in tam.index]
        print(tam.to_frame(name='N').T.to_string())

for nombre_bloque, vars_bloque in grupos_informe_ampliado.items():
            v_num = [v for v in vars_bloque if v in df_desc.columns and pd.api.types.is_numeric_dtype(df_desc[v])]
            v_cat = [v for v in vars_bloque if v in df_desc.columns and not pd.api.types.is_numeric_dtype(df_desc[v])]
            
            # Forzar CUALQUIER variable relacionada con antecedentes a ser categórica (v_cat)
            to_remove = []
            for v in v_num:
                if 'ant_' in v.lower() or 'antecedente' in v.lower() or v == 'fuma' or v == 'alcohol' or v == 'drogas':
                    to_remove.append(v)
            for v in to_remove:
                v_num.remove(v)
                if v not in v_cat:
                    v_cat.append(v)
                    
            if v_num or v_cat:
                print(f"
{'='*40}")
                print(f">>> {nombre_bloque}")
                print(f"{'='*40}")
                
                # Tabla Descriptiva
                print("
[1] Tabla Descriptiva:")
                tabla_desc = describir_clusters(df_desc, v_num, v_cat)
                display(tabla_desc)
                
                # Tabla Estadística
                print("
[2] Análisis Estadístico Completo (P-valores dos a dos):")
                data_rows = []
                for v in (v_num + v_cat):
                    res = realizar_analisis_avanzado(df_desc, v, 'cluster')
                    if res:
                        data_rows.append(res)
                
                if data_rows:
                    k_actual_table = len(df_desc['cluster'].unique())
                    headers = ["Variable", "Test", "p-valor (Total)", "p-valor (dos a dos)"] + [f"G{i}" for i in sorted(df_desc['cluster'].unique())]
                    df_estad = pd.DataFrame(data_rows, columns=headers)
                    display(df_estad)

print("\n----- FIN BLOQUE AVANZADO -----")
